In [1]:
import pandas as pd
import numpy as np
X_train = pd.read_csv('data/X_train.csv')
X_test = pd.read_csv('data/X_test.csv')
y_train = pd.read_csv('data/y_train.csv')['Dataset']
y_test = pd.read_csv('data/y_test.csv')['Dataset']
print('Dados carregados')


Dados carregados


---
# Fase 5 — Pipeline de Pré-processamento

**Objetivo:** Construir pipelines reprodutíveis utilizando ferramentas do scikit-learn que encapsulem todas as transformações de pré-processamento, garantindo ausência de data leakage e consistência entre treino e teste.

Componentes:
- `SimpleImputer` — imputação de valores ausentes (mediana)
- `StandardScaler` — padronização das features
- `Pipeline` — encadeamento de transformações
- `ColumnTransformer` — aplicação de transformações diferentes para colunas diferentes

## 5.1 — Identificação das Colunas

Como `Gender` já foi codificado na Fase 3 (Male=1, Female=0), todas as features agora são numéricas. Ainda assim, separaremos as colunas em dois grupos para flexibilidade:

- **Colunas que precisam de imputação + padronização:** todas as features numéricas contínuas.
- `Gender` já é binária (0/1) e não possui valores ausentes — será incluída no fluxo numérico.

In [2]:
# Identificar colunas
todas_colunas = list(X_train.columns)
print(f'Features ({len(todas_colunas)}): {todas_colunas}')
print(f'\nTipos de dados:')
print(X_train.dtypes.to_string())
print(f'\nValores ausentes em X_train:')
ausentes_treino = X_train.isnull().sum()
print(ausentes_treino[ausentes_treino > 0] if ausentes_treino.sum() > 0 else '  Nenhum (exceto Albumin_and_Globulin_Ratio com poucos NaN)')
print(f'  Total: {ausentes_treino.sum()}')

Features (10): ['Age', 'Gender', 'Total_Bilirubin', 'Direct_Bilirubin', 'Alkaline_Phosphotase', 'Alamine_Aminotransferase', 'Aspartate_Aminotransferase', 'Total_Protiens', 'Albumin', 'Albumin_and_Globulin_Ratio']

Tipos de dados:
Age                             int64
Gender                          int64
Total_Bilirubin               float64
Direct_Bilirubin              float64
Alkaline_Phosphotase            int64
Alamine_Aminotransferase        int64
Aspartate_Aminotransferase      int64
Total_Protiens                float64
Albumin                       float64
Albumin_and_Globulin_Ratio    float64

Valores ausentes em X_train:
Albumin_and_Globulin_Ratio    4
dtype: int64
  Total: 4


## 5.2 — Construção do Preprocessor

Criaremos **dois preprocessors** para dar flexibilidade na modelagem:

1. **`preprocessor_scaled`** — imputação + padronização (`StandardScaler`). Para modelos sensíveis à escala:
   - Regressão Logística, KNN, SVM, **MLP**

2. **`preprocessor_unscaled`** — apenas imputação, sem padronização. Para modelos baseados em árvore:
   - Random Forest, XGBoost

Ambos utilizam `SimpleImputer(strategy='median')` para tratar os valores ausentes em `Albumin_and_Globulin_Ratio`.

In [3]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Lista de todas as features (todas numéricas após a Fase 3)
colunas_numericas = list(X_train.columns)

# --- Preprocessor COM padronização (para modelos sensíveis à escala) ---
pipeline_scaled = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

preprocessor_scaled = ColumnTransformer(transformers=[
    ('num', pipeline_scaled, colunas_numericas)
])

# --- Preprocessor SEM padronização (para modelos baseados em árvore) ---
pipeline_unscaled = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

preprocessor_unscaled = ColumnTransformer(transformers=[
    ('num', pipeline_unscaled, colunas_numericas)
])

print('\u2713 Preprocessors criados:')
print('  - preprocessor_scaled:   Imputação (mediana) + StandardScaler')
print('  - preprocessor_unscaled: Imputação (mediana) apenas')

✓ Preprocessors criados:
  - preprocessor_scaled:   Imputação (mediana) + StandardScaler
  - preprocessor_unscaled: Imputação (mediana) apenas


## 5.3 — Validação do Preprocessor

Testar o preprocessor nos dados de treinamento para verificar que funciona corretamente.

In [4]:
# Testar preprocessor_scaled: fit nos dados de treino, transform em treino e teste
X_train_scaled = preprocessor_scaled.fit_transform(X_train)
X_test_scaled = preprocessor_scaled.transform(X_test)

print('Preprocessor COM padronização:')
print(f'  X_train_scaled shape: {X_train_scaled.shape}')
print(f'  X_test_scaled shape:  {X_test_scaled.shape}')
print(f'  Valores ausentes no treino: {np.isnan(X_train_scaled).sum()}')
print(f'  Valores ausentes no teste:  {np.isnan(X_test_scaled).sum()}')
print(f'\n  Média das features (treino):  {X_train_scaled.mean(axis=0).round(6)}')
print(f'  Desvio padrão (treino):       {X_train_scaled.std(axis=0).round(6)}')
print('  \u2713 Média \u2248 0 e desvio padrão \u2248 1 no treino \u2014 padronização OK.')

Preprocessor COM padronização:
  X_train_scaled shape: (466, 10)
  X_test_scaled shape:  (117, 10)
  Valores ausentes no treino: 0
  Valores ausentes no teste:  0

  Média das features (treino):  [-0.  0. -0.  0. -0. -0.  0. -0. -0.  0.]
  Desvio padrão (treino):       [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
  ✓ Média ≈ 0 e desvio padrão ≈ 1 no treino — padronização OK.


In [5]:
# Testar preprocessor_unscaled
X_train_unscaled = preprocessor_unscaled.fit_transform(X_train)
X_test_unscaled = preprocessor_unscaled.transform(X_test)

print('Preprocessor SEM padronização:')
print(f'  X_train_unscaled shape: {X_train_unscaled.shape}')
print(f'  X_test_unscaled shape:  {X_test_unscaled.shape}')
print(f'  Valores ausentes no treino: {np.isnan(X_train_unscaled).sum()}')
print(f'  Valores ausentes no teste:  {np.isnan(X_test_unscaled).sum()}')
print('  \u2713 Imputação aplicada, escala original mantida.')

Preprocessor SEM padronização:
  X_train_unscaled shape: (466, 10)
  X_test_unscaled shape:  (117, 10)
  Valores ausentes no treino: 0
  Valores ausentes no teste:  0
  ✓ Imputação aplicada, escala original mantida.


## 5.4 — Pipelines Completas (Preprocessor + Modelo)

A pipeline completa encadeia o preprocessor com o classificador, garantindo que:
- `.fit()` ajusta o preprocessor **e** o modelo usando apenas os dados de treinamento.
- `.predict()` aplica as mesmas transformações aprendidas no treino e gera previsões.
- **Data leakage é eliminado** automaticamente.

### Exemplo de uso

```python
# Para modelos que precisam de padronização
pipeline_lr = Pipeline(steps=[
    ('preprocessor', preprocessor_scaled),
    ('classifier', LogisticRegression())
])

# Para modelos baseados em árvore
pipeline_rf = Pipeline(steps=[
    ('preprocessor', preprocessor_unscaled),
    ('classifier', RandomForestClassifier())
])

# Treinar e avaliar
pipeline_lr.fit(X_train, y_train)
y_pred = pipeline_lr.predict(X_test)
```

In [6]:
# Função auxiliar para criar pipelines completas
def criar_pipeline(modelo, usar_scaler=True):
    """
    Cria uma pipeline completa (preprocessor + modelo).
    
    Parâmetros:
        modelo: estimador scikit-learn
        usar_scaler: True para modelos sensíveis à escala,
                     False para modelos baseados em árvore.
    
    Retorna:
        Pipeline completa pronta para .fit() e .predict()
    """
    if usar_scaler:
        pre = ColumnTransformer(transformers=[
            ('num', Pipeline(steps=[
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())
            ]), colunas_numericas)
        ])
    else:
        pre = ColumnTransformer(transformers=[
            ('num', Pipeline(steps=[
                ('imputer', SimpleImputer(strategy='median'))
            ]), colunas_numericas)
        ])
    
    return Pipeline(steps=[
        ('preprocessor', pre),
        ('classifier', modelo)
    ])

print('\u2713 Função criar_pipeline() definida.')
print('  Uso: criar_pipeline(modelo, usar_scaler=True/False)')

✓ Função criar_pipeline() definida.
  Uso: criar_pipeline(modelo, usar_scaler=True/False)


## 5.5 — Resumo: Qual Preprocessor para Cada Modelo?

| Modelo | Preprocessor | Justificativa |
|---|---|---|
| **Regressão Logística** | `preprocessor_scaled` | Regularização (L1/L2) é sensível à escala. Sem padronização, features com valores maiores dominam os coeficientes. |
| **KNN** | `preprocessor_scaled` | Cálculo de distância (euclidiana) é diretamente afetado pela escala. Features com amplitudes maiores dominam a distância. |
| **SVM** | `preprocessor_scaled` | Kernel RBF e outros kernels dependem de distância. Padronização é essencial para convergência e desempenho. |
| **MLP** | `preprocessor_scaled` | Redes neurais utilizam gradiente descendente para otimização dos pesos. Features com escalas diferentes fazem o treinamento convergir lentamente ou ficar instável. |
| **Random Forest** | `preprocessor_unscaled` | Baseado em partições de valores individuais. A escala não afeta a decisão de partição. |
| **XGBoost** | `preprocessor_unscaled` | Mesma justificativa do Random Forest. Invariante à escala das features. |

### Checklist de Conclusão da Fase 5

- ✅ Pipeline numérica implementada (imputação + padronização).
- ✅ `ColumnTransformer` configurado para ambos os preprocessors.
- ✅ `preprocessor_scaled` testado — média ≈ 0, desvio padrão ≈ 1 no treino.
- ✅ `preprocessor_unscaled` testado — imputação OK, escala original mantida.
- ✅ `.fit()` chamado apenas nos dados de treinamento.
- ✅ Dados de teste transformados apenas com `.transform()`.
- ✅ Função `criar_pipeline()` definida para facilitar a criação de pipelines completas.
- ✅ Tabela de correspondência modelo/preprocessor documentada.

In [7]:
import joblib
import os
os.makedirs('models', exist_ok=True)
joblib.dump(preprocessor_scaled, 'models/preprocessor_scaled.joblib')
joblib.dump(preprocessor_unscaled, 'models/preprocessor_unscaled.joblib')
print('Preprocessors salvos em models/')


Preprocessors salvos em models/
